# GBM-Infinite-MC Comparison

## Monte Carlo vs GBM Infinite Analytical Strategy

This notebook compares two withdrawal rate strategies:
1. **MC Strategy**: Monte Carlo simulation with finite time horizon
2. **GBM Infinite Analytical**: Closed-form analytical solution for infinite horizon

We validate both approaches and visualize their behavior across different withdrawal rates.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import gammaincc

# Import FIREworks strategies
from fireworks.strategies import MCStrategy, GBMInfiniteAnalyticStrategy
from fireworks.strategies.mc_strategy import (
    MarketEnvironmentFactory,
    ConsumptionModelFactory,
)

print("Imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# Define market parameters (identical for both strategies)
MU = 0.07  # 7% annual return
VOLATILITY = 0.18  # 18% volatility
VARIANCE = VOLATILITY ** 2  # 0.0324
INITIAL_CAPITAL = 1_000_000  # $1M starting portfolio

# Create market environment and consumption models
market_env = MarketEnvironmentFactory.constant(MU, VARIANCE)
withdrawal_model = ConsumptionModelFactory.constant(40000)  # $40k/year initially

# Initialize strategies
mc_strategy = MCStrategy(
    market_environment=market_env,
    consumption_model=withdrawal_model,
    num_simulations=10000,
    num_steps=100
)

gbm_strategy = GBMInfiniteAnalyticStrategy(
    market_environment=market_env,
    consumption_model=withdrawal_model
)

print(f"Market Parameters:")
print(f"  Mean Return (μ): {MU:.1%}")
print(f"  Volatility (σ): {VOLATILITY:.1%}")
print(f"  Variance (σ²): {VARIANCE:.6f}")
print(f"\nInitial Portfolio: ${INITIAL_CAPITAL:,.0f}")
print(f"Initial Withdrawal: $40,000 (4% rule)")

In [ ]:
# Withdrawal rate grid: 0.5% to 10% in 0.5% increments (centered on 4% rule)
withdrawal_rates = np.arange(0.005, 0.105, 0.005)
num_rates = len(withdrawal_rates)

# Storage
mc_ruin_probs = np.zeros(num_rates)
gbm_ruin_probs = np.zeros(num_rates)

print(f"Withdrawal Rate Grid:")
print(f"  Range: {withdrawal_rates.min():.2%} to {withdrawal_rates.max():.2%}")
print(f"  Increment: 0.5%")
print(f"  Total points: {num_rates}")
print(f"\nComputing sensitivity grid (200-year horizons)...")
for idx, rate in enumerate(withdrawal_rates):
    withdrawal = INITIAL_CAPITAL * rate
    
    # MC Strategy
    mc_result = mc_strategy.simulate(
        initial_capital=INITIAL_CAPITAL,
        annual_withdrawal=withdrawal,
        years=200,
        num_simulations=10000,
        num_steps=12*200  # monthly steps for 200 years
    )
    mc_ruin_probs[idx] = mc_result['ruin_probability']
    
    # GBM Infinite
    gbm_result = gbm_strategy.simulate(
        initial_capital=INITIAL_CAPITAL,
        annual_withdrawal=withdrawal
    )
    gbm_ruin_probs[idx] = gbm_result['ruin_probability']
    
    print(f"  [{idx+1:2d}/{num_rates}] Withdrawal Rate: {rate:.2%} (${withdrawal:,} = MC {mc_ruin_probs[idx]:6.1%} | GBM {gbm_ruin_probs[idx]:6.1%})")

print("\nGrid computation complete!")

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Both strategies
ax1 = axes[0]
ax1.plot(withdrawal_rates * 100, mc_ruin_probs * 100, 'o-', linewidth=2.5, markersize=5, 
         label='MC Strategy (200-year)', color='steelblue', alpha=0.8)
ax1.plot(withdrawal_rates * 100, gbm_ruin_probs * 100, 's-', linewidth=2.5, markersize=5,
         label='GBM Infinite Analytical', color='darkred', alpha=0.8)
ax1.axhline(y=5, color='green', linestyle='--', linewidth=2, alpha=0.6, label='5% Risk Threshold')
ax1.axvline(x=4, color='gray', linestyle=':', linewidth=2, alpha=0.7, label='4% Rule')
ax1.fill_between(withdrawal_rates * 100, 0, 100, where=(withdrawal_rates <= 0.04), 
                  alpha=0.1, color='green', label='Safe Zone')
ax1.set_xlabel('Withdrawal Rate (%)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Probability of Ruin (%)', fontsize=12, fontweight='bold')
ax1.set_title(f'Ruin Probability vs Withdrawal Rate\n(μ=7%, σ=18%)', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left', fontsize=10)
ax1.set_ylim([0, 100])
ax1.set_xlim([0, 10.5])
ax1.set_xticks(np.arange(0, 11, 1))

# Plot 2: Difference (Infinite - 100yr)
ax2 = axes[1]
difference = (gbm_ruin_probs - mc_ruin_probs) * 100
colors = ['red' if d > 0 else 'green' for d in difference]
ax2.plot(withdrawal_rates * 100, difference, 'D-', linewidth=2.5, markersize=5, color='purple', alpha=0.8)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
ax2.axvline(x=4, color='gray', linestyle=':', linewidth=2, alpha=0.7)
ax2.fill_between(withdrawal_rates * 100, 0, difference, where=(difference >= 0), 
                  alpha=0.3, color='red', label='Infinite > 200yr')
ax2.fill_between(withdrawal_rates * 100, 0, difference, where=(difference < 0), 
                  alpha=0.3, color='green', label='200yr > Infinite')
ax2.set_xlabel('Withdrawal Rate (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Ruin Difference (%): GBM - MC', fontsize=12, fontweight='bold')
ax2.set_title('Infinite vs 200-Year Horizon Gap', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left', fontsize=10)
ax2.set_xlim([0, 10.5])
ax2.set_xticks(np.arange(0, 11, 1))

plt.tight_layout()
plt.savefig('fireworks_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved as: fireworks_strategy_comparison.png")